In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

ROOT_DIR = os.getenv("ROOT_DIR")

df_all = pd.read_csv(os.path.join(ROOT_DIR, ".data/processed/processed_all_timeseries.csv"))

In [2]:
df_all['global_id'] = pd.factorize(tuple(zip(df_all['course_id'], df_all['student_id'])))[0]
print(f"Numero di studenti globalmente unici: {df_all['global_id'].nunique()}")

df_all = df_all.sort_values(by=['global_id', 'day']).reset_index(drop=True)

Numero di studenti globalmente unici: 5156


In [3]:
df_all.columns

Index(['student_id', 'course_id', 'day', 'dropout', 'view', 'write',
       'user report', 'update', 'view forum', 'subscribe',
       ...
       'unblocked', 'disabled', 'removed', 'accepted', 'assigned', 'restored',
       'unassigned', 'abandoned', 'recent', 'global_id'],
      dtype='object', length=102)

In [4]:
df_all.head()

,student_id,course_id,day,dropout,view,write,user report,update,view forum,subscribe,...,unblocked,disabled,removed,accepted,assigned,restored,unassigned,abandoned,recent,global_id
0,0,1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0,1,2,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0,1,3,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0,1,4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,0,1,5,1.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [5]:
df_all = df_all.drop(columns=['student_id', 'course_id'])

colonne_restanti = [col for col in df_all.columns if col != 'global_id']
df_all = df_all[['global_id'] + colonne_restanti]

df_all['dropout'] = (df_all['dropout'] >= 0.5).astype(int)

In [6]:
df_all.head()

,global_id,day,dropout,view,write,user report,update,view forum,subscribe,view forums,...,blocked,unblocked,disabled,removed,accepted,assigned,restored,unassigned,abandoned,recent
0,0,1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,2,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,3,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,4,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,5,1,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
df_all.tail()

,global_id,day,dropout,view,write,user report,update,view forum,subscribe,view forums,...,blocked,unblocked,disabled,removed,accepted,assigned,restored,unassigned,abandoned,recent
1856155,5155,356,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1856156,5155,357,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1856157,5155,358,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1856158,5155,359,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1856159,5155,360,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
student_profiles = df_all.groupby('global_id')['dropout'].max().reset_index()

student_profiles['dropout'] = (student_profiles['dropout'] >= 0.5).astype(int)

dist_orig = student_profiles['dropout'].value_counts(normalize=True) * 100
print(f"[INFO] Total unique: {len(student_profiles)}/5156")
print(f"[INFO] Distribution -> Dropout (1): {dist_orig.get(1, 0):.2f}% | Persist (0): {dist_orig.get(0, 0):.2f}%")

[INFO] Total unique: 5156/5156
[INFO] Distribution -> Dropout (1): 79.58% | Persist (0): 20.42%


In [9]:
train_profiles, test_profiles = train_test_split(
    student_profiles,
    test_size=0.20,
    random_state=42,
    stratify=student_profiles['dropout'] 
)

train_ids = train_profiles['global_id']
test_ids = test_profiles['global_id']

In [10]:
df_train = df_all[df_all['global_id'].isin(train_ids)].copy()
df_test  = df_all[df_all['global_id'].isin(test_ids)].copy()

print(f"[INFO] Shape Train: {df_train.shape}")
print(f"[INFO] Shape Test: {df_test.shape}")

[INFO] Shape Train: (1484640, 100)
[INFO] Shape Test: (371520, 100)


In [11]:
# Check 1: Zero Leakage
overlap = set(df_train['global_id']) & set(df_test['global_id'])
assert len(overlap) == 0, f"[FATAL] {len(overlap)} ID crossed"
print("[OK] Leakage Check: Succeded")

[OK] Leakage Check: Succeded


In [12]:
# Check 2: Verifica della stratificazione post-espansione
train_dist = df_train.groupby('global_id')['dropout'].max().value_counts(normalize=True) * 100
test_dist  = df_test.groupby('global_id')['dropout'].max().value_counts(normalize=True) * 100

print(f"[OK] Distribuzione in Train -> Dropout (1): {train_dist.get(1, 0):.2f}% | Persist (0): {train_dist.get(0, 0):.2f}%")
print(f"[OK] Distribuzione in Test  -> Dropout (1): {test_dist.get(1, 0):.2f}% | Persist (0): {test_dist.get(0, 0):.2f}%")

[OK] Distribuzione in Train -> Dropout (1): 79.58% | Persist (0): 20.42%
[OK] Distribuzione in Test  -> Dropout (1): 79.55% | Persist (0): 20.45%


In [14]:
OUTPUT_DIR = os.path.join(ROOT_DIR, ".data/test_train")
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_train.to_csv(os.path.join(OUTPUT_DIR, "train_timeseries.csv"), index=False)
df_test.to_csv(os.path.join(OUTPUT_DIR, "test_timeseries.csv"), index=False)